In [1]:
"""
GDR to CKAN Uploader
====================

A standalone script to upload processed datasets to a CKAN instance.
It traverses the 'gdr_data/files' directory, validates viability via 
'metadata_analysis.json', and aggregates all analysis data.

Modifications:
- Automatically detects compressed files (zip, tar, gz, etc.).
- Uploads the original compressed file.
- Extracts contents to a temporary directory.
- Uploads the individual data files found within the archive.
- Cleans up temporary directories automatically.

Requirements:
    pip install ckanapi
"""

import os
import json
import logging
import re
import shutil
import tempfile
from pathlib import Path
from typing import List, Dict, Any, Optional
import urllib3
import requests

try:
    from ckanapi import RemoteCKAN, NotFound, ValidationError
except ImportError:
    print("CRITICAL: Library 'ckanapi' is required. Run: pip install ckanapi")
    exit(1)

# ------------------------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------------------------

from dotenv import load_dotenv
load_dotenv()

# Environment Variables
CKAN_URL = os.environ.get("CKAN_URL")
CKAN_API_KEY = os.environ.get("CKAN_API_KEY")
CKAN_ORG_ID = os.environ.get("CKAN_ORG_ID")

# Path Configuration
BASE_DIR = Path("gdr_data/files")

# Upload Limits (in Bytes) - Default ~2.4GB
MAX_FILE_SIZE_BYTES = 2400 * 1024 * 1024 

# Extensions considered as archives to be unpacked
COMPRESSED_EXTENSIONS = {'.zip', '.tar', '.gz', '.tgz', '.bz2', '.7z', '.rar'}

# Logging Configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("CKAN_Uploader")

# Suppress InsecureRequestWarning caused by disabling SSL verification
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ------------------------------------------------------------------------------
# DATA FORMATTING HELPERS
# ------------------------------------------------------------------------------

def clean_tag_text(tag: str) -> str:
    """
    Sanitizes tag text for CKAN.
    """
    if not tag:
        return ""
    cleaned = re.sub(r'[^a-zA-Z0-9 \.\-_]', ' ', tag)
    return re.sub(r'\s+', ' ', cleaned).strip()

def format_tags(keywords: List[str]) -> List[Dict[str, str]]:
    """
    Converts keywords to CKAN tag dicts.
    """
    if not keywords:
        return []
    
    unique_tags = set()
    formatted_list = []

    for k in keywords:
        if k and isinstance(k, str):
            clean = clean_tag_text(k)
            if clean and len(clean) > 1 and clean.lower() not in unique_tags:
                unique_tags.add(clean.lower())
                formatted_list.append({"name": clean})
    
    return formatted_list

def format_author(creators: List[Dict[str, str]]) -> str:
    """
    Flattens creators list to string.
    """
    if not creators:
        return ""
    names = [c.get("name") for c in creators if c.get("name")]
    return "; ".join(names)

def generate_valid_slug(text: str) -> str:
    """
    Generates a valid CKAN package URL slug.
    """
    s = text.lower()
    s = re.sub(r'[^a-z0-9]+', '-', s)
    return s.strip('-')

def construct_rich_description(metadata: Dict[str, Any]) -> str:
    """
    Aggregates metadata into Markdown description.
    """
    invenio = metadata.get("invenio_draft", {})
    analysis = metadata.get("analysis", {})
    file_descs = metadata.get("file_descriptions", {})

    description_parts = [invenio.get("description", "No description provided.")]
    description_parts.append("\n\n---")

    variables = analysis.get("variable_dictionary", {})
    if variables:
        description_parts.append("\n### Identified Variables")
        description_parts.append("The following physical and chemical variables were identified in this dataset:\n")
        for var_name, var_desc in variables.items():
            description_parts.append(f"* **{var_name}**: {var_desc}")
    
    if file_descs:
        description_parts.append("\n### File Descriptions")
        for filename, desc in file_descs.items():
            description_parts.append(f"* **{filename}**: {desc}")

    reasoning = analysis.get("reasoning")
    if reasoning:
        description_parts.append("\n### Analysis Context")
        description_parts.append(f"**Viability Assessment:** {reasoning}")

    return "\n".join(description_parts)

# ------------------------------------------------------------------------------
# UPLOAD HELPERS
# ------------------------------------------------------------------------------

def upload_single_resource(ckan: RemoteCKAN, package_id: str, file_path: Path, description: str = ""):
    """
    Helper function to upload a single file to CKAN.
    """
    try:
        if not file_path.exists():
            return

        file_size = file_path.stat().st_size
        if file_size > MAX_FILE_SIZE_BYTES:
            size_mb = file_size / (1024 * 1024)
            logger.warning(f"  [SKIPPED FILE] '{file_path.name}' is {size_mb:.2f}MB (Limit: {MAX_FILE_SIZE_BYTES/(1024*1024)}MB).")
            return
        
        # Don't upload hidden files (macOS .DS_Store, etc)
        if file_path.name.startswith("."):
            return

        logger.info(f"  Uploading resource '{file_path.name}'...")
        with open(file_path, "rb") as f_obj:
            ckan.action.resource_create(
                package_id=package_id,
                name=file_path.name,
                upload=f_obj,
                description=description
            )
    except Exception as e:
        logger.error(f"  [UPLOAD FAILED] '{file_path.name}': {e}")

def handle_compressed_file(ckan: RemoteCKAN, package_id: str, zip_path: Path):
    """
    1. Uploads the original compressed file.
    2. Extracts it to a temp folder.
    3. Uploads extracted contents.
    4. Cleans up temp folder.
    """
    # 1. Upload the original compressed file first
    upload_single_resource(ckan, package_id, zip_path, description="Original Compressed Archive")

    # 2. Extract and Upload Contents
    logger.info(f"  Extracting archive '{zip_path.name}' for processing...")
    
    with tempfile.TemporaryDirectory() as temp_dir:
        try:
            # Attempt to unpack
            shutil.unpack_archive(str(zip_path), temp_dir)
            
            # Walk through the temp directory to find extracted files
            extracted_count = 0
            temp_path = Path(temp_dir)
            
            # Recursive walk to find all files inside the archive
            for root, dirs, files in os.walk(temp_path):
                for file in files:
                    extracted_file_path = Path(root) / file
                    
                    # Skip MAC OS metadata junk if present
                    if "__MACOSX" in str(extracted_file_path) or file.startswith("."):
                        continue
                        
                    desc = f"Extracted from {zip_path.name}"
                    upload_single_resource(ckan, package_id, extracted_file_path, description=desc)
                    extracted_count += 1
            
            if extracted_count == 0:
                logger.warning(f"  Archive '{zip_path.name}' was empty or contained only hidden files.")
                
        except shutil.ReadError:
            logger.error(f"  [ERROR] Could not extract '{zip_path.name}'. Format might not be supported.")
        except Exception as e:
            logger.error(f"  [ERROR] Processing archive '{zip_path.name}' failed: {e}")
    
    # Context manager handles deletion of temp_dir automatically here

# ------------------------------------------------------------------------------
# CORE LOGIC
# ------------------------------------------------------------------------------

def process_submission(ckan: RemoteCKAN, submission_path: Path):
    """
    Reads a single submission folder and uploads it if viable.
    """
    sub_id = submission_path.name
    metadata_file = submission_path / "analysis_results" / "metadata_analysis.json"

    # 1. Validate Existence
    if not metadata_file.exists():
        if (submission_path / "analysis_results").exists():
            logger.warning(f"ID {sub_id}: [SKIPPED] No 'metadata_analysis.json' found.")
        return

    # 2. Load and Validate JSON
    try:
        with open(metadata_file, "r", encoding="utf-8") as f:
            data = json.load(f)
    except json.JSONDecodeError:
        logger.error(f"ID {sub_id}: [FAILED] Invalid JSON syntax.")
        return

    analysis = data.get("analysis", {})
    if not analysis.get("is_viable", False):
        return

    invenio = data.get("invenio_draft", {})
    if not invenio:
        logger.warning(f"ID {sub_id}: [SKIPPED] Missing 'invenio_draft' data.")
        return

    # 3. Prepare CKAN Package Data
    raw_slug_base = f"gdr-submission-{sub_id}"
    package_slug = generate_valid_slug(raw_slug_base)
    rich_notes = construct_rich_description(data)
    clean_tags = format_tags(invenio.get("keywords", []))

    package_dict = {
        "name": package_slug,
        "title": invenio.get("title", f"GDR Submission {sub_id}"),
        "notes": rich_notes,
        "tags": clean_tags,
        "author": format_author(invenio.get("creators", [])),
        "extras": [
            {"key": "original_submission_id", "value": sub_id},
            {"key": "data_source", "value": "Geothermal Data Repository (Processed)"}
        ]
    }

    if CKAN_ORG_ID:
        package_dict["owner_org"] = CKAN_ORG_ID

    # 4. Create or Update Package
    package_id = None
    try:
        try:
            existing_pkg = ckan.action.package_show(id=package_slug)
            logger.info(f"ID {sub_id}: Updating package '{package_slug}'...")
            package_dict["id"] = existing_pkg["id"]
            package = ckan.action.package_update(**package_dict)
            package_id = package["id"]
        except NotFound:
            logger.info(f"ID {sub_id}: Creating package '{package_slug}'...")
            package = ckan.action.package_create(**package_dict)
            package_id = package["id"]
    except ValidationError as e:
        logger.error(f"ID {sub_id}: [FAILED] Validation Error: {e.error_dict}")
        return
    except Exception as e:
        logger.error(f"ID {sub_id}: [FAILED] API Error: {e}")
        return

    # 5. Process Files
    files_to_process = [
        f for f in submission_path.iterdir()
        if f.is_file() and f.name != "analysis_results" and not f.name.startswith(".")
    ]

    if not files_to_process:
        logger.warning(f"ID {sub_id}: Metadata uploaded, but no files found.")
        return

    logger.info(f"ID {sub_id}: Found {len(files_to_process)} root files.")

    for file_path in files_to_process:
        # Check if it is a compressed archive
        is_archive = False
        # simple extension check
        if ''.join(file_path.suffixes).lower() in COMPRESSED_EXTENSIONS or \
           file_path.suffix.lower() in COMPRESSED_EXTENSIONS:
            is_archive = True

        if is_archive:
            # Handle compression (Upload original + Extract & Upload contents)
            handle_compressed_file(ckan, package_id, file_path)
        else:
            # Standard upload
            upload_single_resource(ckan, package_id, file_path, description=f"Raw data file: {file_path.name}")

    logger.info(f"ID {sub_id}: [SUCCESS] Processing complete.")

# ------------------------------------------------------------------------------
# MAIN EXECUTION
# ------------------------------------------------------------------------------

def main():
    if not CKAN_URL or not CKAN_API_KEY:
        logger.error("Missing Environment Variables (CKAN_URL, CKAN_API_KEY).")
        exit(1)

    ua = "gdr-processor/1.0"
    session = requests.Session()
    session.verify = False # SSL bypass
    
    ckan = RemoteCKAN(CKAN_URL, apikey=CKAN_API_KEY, user_agent=ua, session=session)

    if not BASE_DIR.exists():
        logger.error(f"Data directory '{BASE_DIR}' not found.")
        exit(1)

    logger.info(f"Starting upload process from: {BASE_DIR}")
    
    submission_dirs = [x for x in BASE_DIR.iterdir() if x.is_dir()]
    submission_dirs.sort(key=lambda x: x.name)
    
    if not submission_dirs:
        logger.warning("No submission directories found.")

    for folder in submission_dirs:
        process_submission(ckan, folder)

    logger.info("Batch upload job finished.")

if __name__ == "__main__":
    main()

09:20:35 [INFO] Starting upload process from: gdr_data/files
09:20:35 [INFO] ID 1008_Utah_FORGE_Observation_Well_Data: Creating package 'gdr-submission-1008-utah-forge-observation-well-data'...
09:20:36 [INFO] ID 1008_Utah_FORGE_Observation_Well_Data: Found 2 root files.
09:20:36 [INFO]   Uploading resource 'Roosevelt_Observation_Wells_Data.zip'...
09:20:36 [INFO]   Extracting archive 'Roosevelt_Observation_Wells_Data.zip' for processing...
09:20:36 [INFO]   Uploading resource 'OH-7 Temperature Data.pdf'...
09:20:36 [INFO]   Uploading resource 'OH-5 Temperature Log.xlsx'...
09:20:36 [INFO]   Uploading resource 'OH-5 Temperature Data.pdf'...
09:20:36 [INFO]   Uploading resource 'OH-5 Temperature Data and Plot Confirmationt.ai'...
09:20:37 [INFO]   Uploading resource 'OH-4 Temperature Log.xlsx'...
09:20:37 [INFO]   Uploading resource 'OH-4 Subsurface Temperature Survey (A&S 10 Mar 77).pdf'...
09:20:37 [INFO]   Uploading resource 'OH-4 Graphical Geologic Log Version 2.pdf'...
09:20:37 [IN

In [2]:
import shutil
import json
import os
from pathlib import Path

# ------------------------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------------------------
BASE_DIR = Path("gdr_data/files")

# If True: Deletes folders that do NOT have a metadata_analysis.json file.
# If False: Skips them (Safety Mode).
DELETE_MISSING_METADATA = True 

# ------------------------------------------------------------------------------
# CLEANUP LOGIC
# ------------------------------------------------------------------------------

def delete_unsuitable_submissions():
    if not BASE_DIR.exists():
        print(f"Directory {BASE_DIR} does not exist.")
        return

    print(f"Scanning {BASE_DIR}...\n")
    print(f"Configuration: Delete submissions with missing metadata? {'YES' if DELETE_MISSING_METADATA else 'NO'}")

    deleted_count = 0
    kept_count = 0
    skipped_count = 0

    for submission_dir in BASE_DIR.iterdir():
        if not submission_dir.is_dir():
            continue

        sub_id = submission_dir.name
        metadata_file = submission_dir / "analysis_results" / "metadata_analysis.json"

        # ---------------------------------------------------------
        # CASE 1: Metadata File Missing
        # ---------------------------------------------------------
        if not metadata_file.exists():
            if DELETE_MISSING_METADATA:
                print(f"[DELETE] ID {sub_id}: Metadata missing (Forced Delete).")
                try:
                    shutil.rmtree(submission_dir)
                    deleted_count += 1
                except Exception as e:
                    print(f"    Error deleting: {e}")
            else:
                print(f"[SKIP]   ID {sub_id}: Metadata missing (Skipped).")
                skipped_count += 1
            
            # Stop processing this folder
            continue

        # ---------------------------------------------------------
        # CASE 2: Metadata Exists - Check Viability
        # ---------------------------------------------------------
        try:
            with open(metadata_file, "r", encoding="utf-8") as f:
                data = json.load(f)

            analysis = data.get("analysis", {})
            is_viable = analysis.get("is_viable")
            reasoning = analysis.get("reasoning", "No reason provided")

            if is_viable is False:
                print(f"[DELETE] ID {sub_id}")
                print(f"         Reason: {reasoning}")
                shutil.rmtree(submission_dir)
                deleted_count += 1
            else:
                # is_viable is True or None
                kept_count += 1

        except json.JSONDecodeError:
            print(f"[ERROR]  ID {sub_id}: Corrupt JSON file. Skipping.")
            skipped_count += 1
        except Exception as e:
            print(f"[ERROR]  ID {sub_id}: Unexpected error: {e}")
            skipped_count += 1

    print("-" * 30)
    print("Cleanup Complete.")
    print(f"Deleted: {deleted_count}")
    print(f"Kept:    {kept_count}")
    print(f"Skipped: {skipped_count}")

if __name__ == "__main__":
    delete_unsuitable_submissions()

Scanning gdr_data/files...

Configuration: Delete submissions with missing metadata? YES
[DELETE] ID 1775_Utah_FORGE_Fluid_Injection-Rate_Controls_on_Seismi: Metadata missing (Forced Delete).
[DELETE] ID 1774_Utah_FORGE_Laboratory_Fault_Reactivation_and_Perme: Metadata missing (Forced Delete).
[DELETE] ID 1801_Northwestern_Michigan_College_Formation_Thermal_Co: Metadata missing (Forced Delete).
[DELETE] ID 1702_Renewable_Energy_Potential_Model_Hawaii_Geothermal: Metadata missing (Forced Delete).
[DELETE] ID 1704_Nationwide_Heat_Flow,_Temperature_Gradient,_and_Re: Metadata missing (Forced Delete).
[DELETE] ID 1768_Ionic_Fluids_for_EGS_Thermogravimetric_Analysis_of: Metadata missing (Forced Delete).
[DELETE] ID 1786_Utah_FORGE_6-3656_Real-Time_Traffic_Light_System_a: Metadata missing (Forced Delete).
[DELETE] ID 1727_Utah_FORGE_Well_16B(78)-32_Reinterpretation_of_Thr: Metadata missing (Forced Delete).
[DELETE] ID 1750_Utah_FORGE_Updated_Discrete_Fracture_Network_Model: Metadata missing (